# Tournament Model Comparison: DQN v4 vs SAC
Runs DQN v4 and SAC head-to-head plus each vs minimax depth-3 and random.
Run all cells top to bottom. Drive must be mounted so the model files are accessible.

In [ ]:
# Cell 1: Setup — clone repo, mount Drive, configure TF
import os, sys, random
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus[0].name if gpus else 'None (CPU)')

# Mount Drive — needed to pull the latest weights if they live there
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
except Exception:
    print('Drive not mounted — using repo files only.')

# Clone re_org branch — has both dqn_v4_best.weights.h5 and RL models/sac_zan.keras
REPO_DIR = Path('/content/connect4-rl-arena')
if not REPO_DIR.exists():
    os.system('git clone --branch re_org https://github.com/Stiles-Clements1/connect4-rl-arena.git ' + str(REPO_DIR))
    print('Repo cloned (re_org branch).')
else:
    os.system(f'git -C {REPO_DIR} fetch origin re_org && git -C {REPO_DIR} checkout re_org && git -C {REPO_DIR} pull')
    print('Repo updated to re_org branch.')

sys.path.insert(0, str(REPO_DIR))

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('Setup complete.')

In [ ]:
# Cell 2: Load models
from src import eval as ev
from src.model_loader import ModelWrapper

# ── DQN v4: rebuild architecture, then load weights ──────────────────────────
def build_dueling_dqn(input_shape=(6, 7, 2), n_actions=7, name='online'):
    inp = layers.Input(shape=input_shape, name='board')
    x   = layers.Conv2D(64,  (3,3), padding='same', activation='relu')(inp)
    x   = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    res = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    res = layers.Conv2D(128, (3,3), padding='same')(res)
    x   = layers.Activation('relu')(layers.Add()([x, res]))
    x   = layers.Dense(512, activation='relu')(layers.Flatten()(x))
    v   = layers.Dense(1,   name='value')    (layers.Dense(256, activation='relu')(x))
    a   = layers.Dense(n_actions, name='advantage')(layers.Dense(256, activation='relu')(x))
    q   = layers.Lambda(
        lambda va: va[0] + (va[1] - tf.reduce_mean(va[1], axis=1, keepdims=True)),
        name='q'
    )([v, a])
    return models.Model(inp, q, name=name)

DQN_WEIGHTS = REPO_DIR / 'dqn_v4_best.weights.h5'
dqn_model = build_dueling_dqn(name='dqn_v4')
dqn_model.load_weights(str(DQN_WEIGHTS))
print(f'DQN v4 loaded  ({DQN_WEIGHTS.stat().st_size / 1e6:.1f} MB)')

# ── SAC: full saved model ─────────────────────────────────────────────────────
SAC_PATH = REPO_DIR / 'RL models' / 'sac_zan.keras'
sac_model = tf.keras.models.load_model(str(SAC_PATH), compile=False)
print(f'SAC loaded  input={sac_model.input_shape}  ({SAC_PATH.stat().st_size / 1e6:.1f} MB)')

# Both models use Type B encoding (6,7,2) — Stiles/Zan architecture
dqn_wrapper = ModelWrapper(model=dqn_model, encoding='B', name='dqn_v4')
sac_wrapper = ModelWrapper(model=sac_model, encoding='B', name='sac')

dqn_agent = ev.ModelAgent(dqn_wrapper, greedy=True, use_tactics=True)
sac_agent = ev.ModelAgent(sac_wrapper, greedy=True, use_tactics=True)
mm3_agent = ev.MinimaxAgent(depth=3)
rnd_agent = ev.RandomAgent()

print('\nAgents ready:', [a.name for a in [dqn_agent, sac_agent, mm3_agent, rnd_agent]])

In [ ]:
# Cell 3: Run matchups
# 200 games each, 6 random init moves to diversify starting positions
N    = 200
INIT = 6

print(f'Running {N}-game matches (random_init_moves={INIT})...\n')

r_dqn_vs_sac = ev.play_match(dqn_agent, sac_agent, n_games=N, random_init_moves=INIT)
r_dqn_vs_mm3 = ev.play_match(dqn_agent, mm3_agent, n_games=N, random_init_moves=INIT)
r_sac_vs_mm3 = ev.play_match(sac_agent, mm3_agent, n_games=N, random_init_moves=INIT)
r_dqn_vs_rnd = ev.play_match(dqn_agent, rnd_agent, n_games=N, random_init_moves=INIT)
r_sac_vs_rnd = ev.play_match(sac_agent, rnd_agent, n_games=N, random_init_moves=INIT)

print('\n' + '='*62)
for r in [r_dqn_vs_sac, r_dqn_vs_mm3, r_sac_vs_mm3, r_dqn_vs_rnd, r_sac_vs_rnd]:
    print(ev.format_result(r))
    print()

In [ ]:
# Cell 4: Summary — which model to submit for the tournament?
import matplotlib.pyplot as plt

labels   = ['vs SAC', 'vs Minimax d3', 'vs Random']
dqn_wr   = [r_dqn_vs_sac.a_win_rate, r_dqn_vs_mm3.a_win_rate, r_dqn_vs_rnd.a_win_rate]
sac_wr   = [r_dqn_vs_sac.b_win_rate, r_sac_vs_mm3.a_win_rate, r_sac_vs_rnd.a_win_rate]

x = range(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([i - w/2 for i in x], [v*100 for v in dqn_wr], w, label='DQN v4', color='steelblue')
ax.bar([i + w/2 for i in x], [v*100 for v in sac_wr], w, label='SAC',    color='darkorange')
ax.axhline(50, color='gray', linestyle='--', linewidth=0.8)
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.set_ylabel('Win rate (%)')
ax.set_title(f'DQN v4 vs SAC — Tournament Model Selection ({N} games each)')
ax.legend(); ax.set_ylim(0, 105)
for i, (d, s) in enumerate(zip(dqn_wr, sac_wr)):
    ax.text(i - w/2, d*100 + 1, f'{d:.0%}', ha='center', va='bottom', fontsize=8)
    ax.text(i + w/2, s*100 + 1, f'{s:.0%}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('/content/tournament_model_comparison.png', dpi=150)
plt.show()

dqn_avg = sum(dqn_wr) / len(dqn_wr)
sac_avg = sum(sac_wr) / len(sac_wr)
winner  = 'DQN v4' if dqn_avg >= sac_avg else 'SAC'
print(f'DQN v4 avg win rate: {dqn_avg:.1%}')
print(f'SAC    avg win rate: {sac_avg:.1%}')
print(f'\n→ Recommended tournament submission: {winner}')